# DC-Ops: Fine-tune RetinaNet + Export with QNN HTP for Snapdragon NPU

1. Fine-tune RetinaNet-ResNet50-FPN on 4 Roboflow DC datasets (2,036 images)
2. Export to ExecuTorch .pte with QNN HTP backend for SM8750 (Snapdragon 8 Elite)

**Runtime → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q roboflow torchvision py-cpuinfo

# Clone and install ExecuTorch with QNN
!git clone --depth 1 https://github.com/pytorch/executorch.git
%cd executorch
!git submodule sync
!git submodule update --init --recursive --depth 1
!bash install_requirements.sh
!pip install -e . --no-build-isolation

# Download QNN SDK
!python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null
import os
qnn_path = !python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null
os.environ['QNN_SDK_ROOT'] = qnn_path[-1].strip()
print(f'QNN SDK: {os.environ["QNN_SDK_ROOT"]}')

In [ ]:
# 2. Download Roboflow datasets
import os
os.chdir('/content')

from roboflow import Roboflow
API_KEY = input('Paste your Roboflow API key: ')
rf = Roboflow(api_key=API_KEY)

print('1/4 Ports (ym)...')
rf.workspace('ym-pffnw').project('ports-yutnj').version(1).download('coco', location='roboflow_ports_ym')
print('2/4 Server Detection (Fujitsu)...')
rf.workspace('fujitsu-ih8ei').project('server-detection').version(1).download('coco', location='roboflow_fujitsu')
print('3/4 Server Vision (hunters)...')
rf.workspace('hunters-workspace-kqclz').project('server-vision').version(5).download('coco', location='roboflow_server_vision')
print('4/4 PC Ports...')
rf.workspace('ports-gmmmp').project('pc-ports').version(10).download('coco', location='roboflow_pc_ports')
print('All downloaded!')

In [ ]:
# 3. Merge datasets into COCO format with DC-Ops classes
import json, shutil, glob
from pathlib import Path

DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

# Same class maps as before (source_id -> DC-Ops id)
PORTS_YM_MAP = {0:6, 1:6, 2:6, 3:12, 4:6, 5:6, 6:6}
FUJITSU_MAP = {0:6, 1:6, 2:6, 3:6, 4:6, 5:15, 6:1, 7:1, 8:6}
SERVER_VISION_MAP = {
    0:1, 1:13, 2:1, 3:1, 4:1, 5:1, 6:13, 7:13, 8:13, 9:9,
    10:13, 11:6, 12:6, 13:10, 14:14, 15:6, 16:14, 17:15, 18:13,
    19:12, 20:12, 21:15, 22:4, 23:15, 24:6, 25:15, 26:14, 27:6, 28:6, 29:6
}
PC_PORTS_MAP = {0:6, 1:6, 2:6, 3:6}

DATASETS = [
    ('roboflow_ports_ym', PORTS_YM_MAP),
    ('roboflow_fujitsu', FUJITSU_MAP),
    ('roboflow_server_vision', SERVER_VISION_MAP),
    ('roboflow_pc_ports', PC_PORTS_MAP),
]

# Build merged COCO dataset
merged_dir = Path('merged_coco')
for split in ['train', 'val']:
    (merged_dir / split).mkdir(parents=True, exist_ok=True)

def merge_coco_datasets(split_map):
    """split_map: list of (src_split_name, dst_split_name)"""
    for dst_split in ['train', 'val']:
        all_images = []
        all_annotations = []
        img_id = 0
        ann_id = 0
        
        for ds_path, class_map in DATASETS:
            ds = Path(ds_path)
            for src_split, target_split in [('train', 'train'), ('valid', 'val'), ('test', 'train')]:
                if target_split != dst_split:
                    continue
                ann_file = ds / src_split / '_annotations.coco.json'
                if not ann_file.exists():
                    continue
                
                with open(ann_file) as f:
                    coco = json.load(f)
                
                # Build old_id -> new_id mapping for images
                old_to_new_img = {}
                for img_info in coco.get('images', []):
                    old_id = img_info['id']
                    img_id += 1
                    old_to_new_img[old_id] = img_id
                    
                    # Copy image
                    src_img = ds / src_split / img_info['file_name']
                    new_name = f"{Path(ds_path).name}_{src_split}_{img_info['file_name']}"
                    dst_img = merged_dir / dst_split / new_name
                    if src_img.exists():
                        shutil.copy2(src_img, dst_img)
                    
                    all_images.append({
                        'id': img_id,
                        'file_name': new_name,
                        'width': img_info.get('width', 640),
                        'height': img_info.get('height', 640),
                    })
                
                # Map annotations
                # Build source category id mapping
                src_cat_map = {}
                for cat in coco.get('categories', []):
                    src_cat_map[cat['id']] = cat['name']
                
                # Map source cat ids to sequential 0-based, then to DC-Ops
                sorted_src_ids = sorted(src_cat_map.keys())
                src_id_to_seq = {sid: i for i, sid in enumerate(sorted_src_ids)}
                
                for ann in coco.get('annotations', []):
                    old_cat = ann['category_id']
                    seq_id = src_id_to_seq.get(old_cat)
                    if seq_id is None:
                        continue
                    new_cat = class_map.get(seq_id)
                    if new_cat is None:
                        continue
                    old_img_id = ann['image_id']
                    if old_img_id not in old_to_new_img:
                        continue
                    
                    ann_id += 1
                    all_annotations.append({
                        'id': ann_id,
                        'image_id': old_to_new_img[old_img_id],
                        'category_id': new_cat + 1,  # COCO is 1-indexed
                        'bbox': ann['bbox'],
                        'area': ann.get('area', ann['bbox'][2] * ann['bbox'][3]),
                        'iscrowd': 0,
                    })
        
        # Write merged COCO annotation
        categories = [{'id': i+1, 'name': name, 'supercategory': 'dc_component'}
                      for i, name in enumerate(DC_CLASSES)]
        
        coco_out = {
            'images': all_images,
            'annotations': all_annotations,
            'categories': categories,
        }
        
        ann_path = merged_dir / f'annotations_{dst_split}.json'
        with open(ann_path, 'w') as f:
            json.dump(coco_out, f)
        
        print(f'{dst_split}: {len(all_images)} images, {len(all_annotations)} annotations')

merge_coco_datasets([('train', 'train'), ('valid', 'val'), ('test', 'train')])
print('Merged COCO dataset ready!')

In [ ]:
# 4. Fine-tune RetinaNet on DC-Ops data
import torch
import torchvision
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json, os, time

NUM_CLASSES = 16

class DCOpsDataset(Dataset):
    def __init__(self, img_dir, ann_file):
        with open(ann_file) as f:
            self.coco = json.load(f)
        self.img_dir = img_dir
        self.images = self.coco['images']
        # Group annotations by image
        self.img_anns = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_anns:
                self.img_anns[img_id] = []
            self.img_anns[img_id].append(ann)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((640, 640)),
        ])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        img = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size
        img = self.transform(img)
        
        anns = self.img_anns.get(img_info['id'], [])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            # Scale to 640x640
            sx, sy = 640.0 / orig_w, 640.0 / orig_h
            x1 = x * sx
            y1 = y * sy
            x2 = (x + w) * sx
            y2 = (y + h) * sy
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(ann['category_id'])
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        
        target = {'boxes': boxes, 'labels': labels}
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Load pre-trained RetinaNet and modify head for 16+1 classes
model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
num_anchors = model.head.classification_head.num_anchors
model.head.classification_head.num_classes = NUM_CLASSES + 1  # +1 for background
in_channels = model.head.classification_head.conv[0].in_channels
model.head.classification_head.cls_logits = torch.nn.Conv2d(
    in_channels, num_anchors * (NUM_CLASSES + 1), kernel_size=3, stride=1, padding=1
)

# Datasets
train_ds = DCOpsDataset('merged_coco/train', 'merged_coco/annotations_train.json')
val_ds = DCOpsDataset('merged_coco/val', 'merged_coco/annotations_val.json')
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')
print(f'Classes: {NUM_CLASSES} + background')

# Train
device = torch.device('cuda')
model.to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
start = time.time()
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    for i, (images, targets) in enumerate(train_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()
    
    lr_scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  loss: {avg_loss:.4f}  time: {elapsed/60:.1f}min')

# Save trained model
torch.save(model.state_dict(), 'retinanet_dc_ops.pth')
print(f'\nTraining complete! Saved retinanet_dc_ops.pth')

In [ ]:
# 5. Export to ExecuTorch with QNN HTP backend
import os, sys
os.chdir('/content/executorch')
sys.path.insert(0, '/content/executorch')

import torch
from executorch.backends.qualcomm.export_utils import (
    build_executorch_binary,
    QnnConfig,
)
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype
from executorch.backends.qualcomm.serialization.qc_schema import (
    QnnExecuTorchBackendType,
    QcomChipset,
)
import torchvision
from torchvision.models.detection import retinanet_resnet50_fpn_v2

NUM_CLASSES = 16

# Reload model with trained weights
model = retinanet_resnet50_fpn_v2(weights=None)
num_anchors = model.head.classification_head.num_anchors
model.head.classification_head.num_classes = NUM_CLASSES + 1
in_channels = model.head.classification_head.conv[0].in_channels
model.head.classification_head.cls_logits = torch.nn.Conv2d(
    in_channels, num_anchors * (NUM_CLASSES + 1), kernel_size=3, stride=1, padding=1
)
model.load_state_dict(torch.load('/content/retinanet_dc_ops.pth', map_location='cpu'))

# Strip post-processing for export (same as the example script)
def forward_without_metrics(self, image):
    features = self.backbone(image)
    return self.head(list(features.values()))

model.forward = lambda img: forward_without_metrics(model, img)
model.eval()

# Calibration data
calib_inputs = [(torch.randn(1, 3, 640, 640),) for _ in range(20)]

# QNN config for SM8750 HTP
qnn_config = QnnConfig(
    soc_model=QcomChipset.SM8750,
    backend=QnnExecuTorchBackendType.kHtpBackend,
)

print('Building ExecuTorch binary with QNN HTP backend...')
print('Target: SM8750 (Snapdragon 8 Elite) Hexagon HTP')
build_executorch_binary(
    model=model,
    qnn_config=qnn_config,
    file_name='/content/dc_ops_retinanet_qnn',
    dataset=calib_inputs,
    quant_dtype=QuantDtype.use_8a8w,
)

pte_path = '/content/dc_ops_retinanet_qnn.pte'
print(f'\nExported: {pte_path} ({os.path.getsize(pte_path) / 1e6:.1f} MB)')
print('Backend: QNN HTP (Hexagon Tensor Processor)')
print('Quantization: INT8 (8a8w)')
print('Target SoC: SM8750 (Snapdragon 8 Elite)')

In [ ]:
# 6. Upload to HuggingFace and download
!pip install -q huggingface_hub
from huggingface_hub import HfApi
api = HfApi()

# Upload .pte
api.upload_file(
    path_or_fileobj='/content/dc_ops_retinanet_qnn.pte',
    path_in_repo='models/dc_ops_retinanet_qnn.pte',
    repo_id='abhijitbetigeri/dc-ops-dataset',
    repo_type='dataset',
    commit_message='Add RetinaNet QNN HTP .pte for Snapdragon 8 Elite NPU',
)

# Upload .pth weights
api.upload_file(
    path_or_fileobj='/content/retinanet_dc_ops.pth',
    path_in_repo='models/retinanet_dc_ops.pth',
    repo_id='abhijitbetigeri/dc-ops-dataset',
    repo_type='dataset',
    commit_message='Add RetinaNet DC-Ops trained weights',
)

print('Uploaded to HuggingFace!')

from google.colab import files
files.download('/content/dc_ops_retinanet_qnn.pte')